> The monitoring data and building-specific configuration are confidential and are not included. See `README.md` for execution requirements. Clear any generated outputs before sharing an executed notebook.


In [ ]:
# Private workspace and configuration
from copy import deepcopy as _privacy_deepcopy
from functools import lru_cache as _privacy_lru_cache
import json
import os
from pathlib import Path


def _private_config_path(variable):
    value = os.environ.get(variable)
    if not value:
        raise RuntimeError(f"Set {variable} before running this notebook; see README.md.")
    return Path(value).expanduser().resolve()


def private_workspace():
    """Return the configured private input/output directory."""
    path = _private_config_path("HVAC_PRIVATE_WORKSPACE")
    if not path.is_dir():
        raise FileNotFoundError("The configured private workspace does not exist.")
    return path


def _decode_private_values(value):
    if isinstance(value, list):
        return [_decode_private_values(item) for item in value]
    if isinstance(value, dict):
        if set(value) == {"__tuple_key_dict__"}:
            return {tuple(key): _decode_private_values(item)
                    for key, item in value["__tuple_key_dict__"]}
        return {key: _decode_private_values(item) for key, item in value.items()}
    return value


@_privacy_lru_cache(maxsize=1)
def _load_private_values():
    with _private_config_path("HVAC_PRIVATE_SETTINGS").open(encoding="utf-8") as handle:
        values = json.load(handle).get("private_values")
    if not isinstance(values, dict):
        raise ValueError("Private settings must contain a private_values object.")
    return _decode_private_values(values)


def private_value(key):
    """Read a private setting without embedding its value in the public source."""
    values = _load_private_values()
    if key not in values or values[key] is None:
        raise KeyError(f"Required private setting is missing: {key!r}.")
    return _privacy_deepcopy(values[key])


# Notebook 04: Sampling uncertainty and prediction coverage

This notebook analyzes the saved predictions from Notebook 03. It verifies the input run and chronological split, reconstructs the common test cases, and computes prediction-time and event-averaged errors. Models, labels and predictions remain fixed.

An **anchor** identifies one prediction time. Common test cases contain a finite prediction from every compared model. Prediction-time averages weight these rows equally; event averages first summarize each event and then weight events equally.

Paired bootstrap comparisons resample whole events with replacement, retaining all prediction times and using the same sampled events for each model. A second analysis resamples within rooms while preserving room-specific event counts. Results also describe excluded cases and the fractions of predictions within specified timing tolerances.

These intervals describe sampling variation within the observed building and selected test events. They do not provide cross-building validation or calibrated uncertainty for an individual prediction. Events from different rooms on the same day are not jointly resampled as a daily block.


In [ ]:


# Imports, private paths, and Notebook-04 configuration

import os, re, json, math, hashlib, warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

try:
    from google.colab import drive  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
except Exception:
    pass

# Reference locations used by Notebooks 01–03

BASE_DIR = Path(str(private_workspace()))
DATA_DIR = BASE_DIR / 'full_rampup_5min_feature_dataset_UNIFIED_DERIVATIVE_DURATION_AWARE'
NB03_ROOT = BASE_DIR / 'duration_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL'

# Select the saved evaluation run through the private configuration.
NOTEBOOK03_RUN_ID = private_value('shared.notebook03_run_id')
NB03_RUN = NB03_ROOT / NOTEBOOK03_RUN_ID

# Notebook 04 never overwrites Notebook 03.
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
OUT_ROOT = BASE_DIR / 'robustness_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL' / RUN_ID
TABLE_DIR = OUT_ROOT / 'tables'
FIG_DIR = OUT_ROOT / 'figures'
AUDIT_DIR = OUT_ROOT / 'audit'
for p in [TABLE_DIR, FIG_DIR, AUDIT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Statistical configuration

METHODS = [
    'Direct_RoomPhase_ET',
    'Indirect_RoomPhase_ET',
    'Unified_Derivative_NN',
]
PHASES = ['heating', 'cooling']
STEP_MIN = 5.0

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 42
CI_LEVEL = 0.95
TOLERANCES_MIN = [5.0, 10.0, 15.0, 20.0]

# Run a second bootstrap that preserves room composition by resampling
# events within each room.
RUN_ROOM_STRATIFIED_BOOTSTRAP = True

# Stop if input counts or metrics disagree with the saved reference expectations.
STRICT_REFERENCE_AUDIT = True

print('Notebook 03 run:', NB03_RUN)
print('Notebook 04 output:', OUT_ROOT)
print('Bootstrap replicates:', N_BOOTSTRAP)


## 1. Verify the input run and split

Recompute the split fingerprint from `phase`, `room`, `event_id`, `split` and `split_block`, and compare it with the saved training and evaluation records. These checks detect inconsistent inputs before additional statistics are calculated.


In [ ]:


# Integrity helpers and exact Notebook-03 run verification

def norm_event_id(x):
    if pd.isna(x):
        raise RuntimeError('Missing event_id')
    s = str(x).strip()
    if re.fullmatch(r'[-+]?\d+\.0+', s):
        s = str(int(float(s)))
    return s

def shared_split_sha256(path):
    z = pd.read_csv(path)
    z['room'] = z['room'].astype(str).str.strip()
    z['event_id'] = z['event_id'].map(norm_event_id)
    cols = ['phase', 'room', 'event_id', 'split', 'split_block']
    z = (
        z[cols]
        .drop_duplicates()
        .sort_values(['phase', 'room', 'event_id'])
        .reset_index(drop=True)
    )
    payload = z.to_csv(index=False, lineterminator='\n').encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

SPLIT_PATH = DATA_DIR / 'shared_event_split_map.csv'
PRED_PATH = NB03_RUN / 'predictions' / 'ALL_remaining_duration_predictions_with_relative_errors.csv'
NB03_MAIN_PATH = NB03_RUN / 'tables' / 'PAPER_MAIN_duration_metrics_EXACT_COMMON_SUPPORT.csv'
NB03_MANIFEST_PATH = NB03_RUN / 'duration_manifest_UNIFIED_DERIVATIVE_DURATION_AWARE_PAPER_FINAL.json'

required_paths = [SPLIT_PATH, PRED_PATH, NB03_MAIN_PATH, NB03_MANIFEST_PATH]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Notebook 04 requires the exact completed Notebook 03 run. Missing:\n' +
        '\n'.join(missing)
    )

CURRENT_SPLIT_SHA256 = shared_split_sha256(SPLIT_PATH)
with open(NB03_MANIFEST_PATH, 'r', encoding='utf-8') as f:
    NB03_MANIFEST = json.load(f)

if NB03_MANIFEST.get('shared_split_sha256') != CURRENT_SPLIT_SHA256:
    raise RuntimeError(
        'Notebook 03 manifest and the current authoritative split have different SHA256 fingerprints.'
    )

if list(NB03_MANIFEST.get('methods', [])) != METHODS:
    raise RuntimeError(
        f'Notebook 03 method list differs from PAPER-FINAL methods: {NB03_MANIFEST.get("methods")}'
    )

if str(NB03_MANIFEST.get('primary_endpoint', '')).strip().lower() != 'remaining duration':
    raise RuntimeError('Notebook 03 manifest does not identify remaining duration as the primary endpoint.')

INPUT_PREDICTION_SHA256 = sha256_file(PRED_PATH)

print('Shared split SHA256:', CURRENT_SPLIT_SHA256)
print('Prediction CSV SHA256:', INPUT_PREDICTION_SHA256)
print('Notebook 03 manifest integrity: PASSED')


## 2. Load test prediction rows

Reconstruct duration anchors using Notebook 03's event identifiers, split assignments, prediction keys, observed durations and regularity scores. Total event duration is added for coverage diagnostics only.


In [ ]:


# Load saved anchor metadata exactly as Notebook 03

split = pd.read_csv(SPLIT_PATH)
split['event_id'] = split['event_id'].map(norm_event_id)
split['room'] = split['room'].astype(str).str.strip()

if 'shared_event_uid' not in split.columns:
    split['shared_event_uid'] = (
        split['phase'].astype(str) + '|' +
        split['room'].astype(str) + '|' +
        split['event_id']
    )

split['event_key'] = (
    split['phase'].astype(str) + '|' +
    split['room'].astype(str) + '|' +
    split['event_id']
)

split_small = split[
    ['event_key', 'shared_event_uid', 'split', 'split_block']
].drop_duplicates()

anchor_frames = []
for phase in PHASES:
    p = DATA_DIR / phase / f'{phase}_remaining_duration_anchors_5min.csv'
    if not p.exists():
        raise FileNotFoundError(p)

    d = pd.read_csv(p)
    d['phase'] = phase

    for c in [
        'timestamp', 'anchor_timestamp', 'feature_source_timestamp',
        'rampup_start_time', 'rampup_end_time', 'rampup_end_grid_time'
    ]:
        if c in d.columns:
            d[c] = pd.to_datetime(d[c], errors='coerce')

    d['event_id'] = d['event_id'].map(norm_event_id)
    d['room'] = d['room'].astype(str).str.strip()
    d['event_key'] = d['phase'] + '|' + d['room'] + '|' + d['event_id']

    d = d.merge(split_small, on='event_key', how='left', validate='many_to_one')
    if d['split'].isna().any():
        raise RuntimeError(f'{phase}: unmatched anchors in authoritative shared split.')

    d['true_remaining_duration_min'] = pd.to_numeric(
        d.get('true_remaining_duration_min', d.get('remaining_duration_min')),
        errors='coerce'
    )
    d['remaining_trajectory_quality_score'] = pd.to_numeric(
        d['remaining_trajectory_quality_score'], errors='coerce'
    ).clip(0.0, 1.0)

    anchor_kind = (
        d['anchor_kind']
        if 'anchor_kind' in d.columns
        else pd.Series('anchor', index=d.index)
    ).astype(str)

    d['anchor_key'] = (
        d['event_key'] + '|' +
        d['anchor_timestamp'].astype(str) + '|' +
        anchor_kind
    )

    # Event duration is used only for support-selection diagnostics.
    if 'true_total_duration_min' in d.columns:
        d['event_duration_min'] = pd.to_numeric(d['true_total_duration_min'], errors='coerce')
    elif {'rampup_start_time', 'rampup_end_time'}.issubset(d.columns):
        d['event_duration_min'] = (
            d['rampup_end_time'] - d['rampup_start_time']
        ).dt.total_seconds() / 60.0
    else:
        d['event_duration_min'] = np.nan

    anchor_frames.append(d)

ANCHORS = pd.concat(anchor_frames, ignore_index=True, sort=False)
TEST_ANCHORS = ANCHORS[ANCHORS['split'].eq('test')].copy()

if TEST_ANCHORS['anchor_key'].duplicated().any():
    raise RuntimeError('Authoritative TEST anchor_key must be unique.')

# Fixed reference counts from the executed reference notebooks.
EXPECTED_FULL_TEST = private_value('nb04.expected_full_test')

full_counts = (
    TEST_ANCHORS.groupby('phase')
    .agg(
        anchors=('anchor_key', 'nunique'),
        events=('shared_event_uid', 'nunique'),
        rooms=('room', 'nunique'),
    )
    .reset_index()
)

print('Authoritative TEST support:')
display(full_counts)

if STRICT_REFERENCE_AUDIT:
    for _, r in full_counts.iterrows():
        phase = r['phase']
        exp = EXPECTED_FULL_TEST[phase]
        if int(r['anchors']) != exp['anchors'] or int(r['events']) != exp['events']:
            raise RuntimeError(
                f'{phase}: TEST support differs from frozen reference. '
                f'Observed anchors/events={int(r["anchors"])}/{int(r["events"])}, '
                f'expected={exp["anchors"]}/{exp["events"]}.'
            )
    print('Frozen TEST-count audit: PASSED')


In [ ]:


# Load Notebook 03 predictions and independently reconstruct
#    exact-common TEST support

ALL = pd.read_csv(PRED_PATH)

required_pred_cols = [
    'anchor_key', 'phase', 'room', 'event_key', 'split', 'anchor_kind',
    'true_remaining_duration_min', 'remaining_trajectory_quality_score',
    'method', 'pred_remaining_duration_min'
]
missing_cols = [c for c in required_pred_cols if c not in ALL.columns]
if missing_cols:
    raise RuntimeError(f'Notebook 03 prediction CSV is missing columns: {missing_cols}')

for c in [
    'true_remaining_duration_min', 'remaining_trajectory_quality_score',
    'pred_remaining_duration_min', 'error_min', 'absolute_error_min',
    'relative_absolute_error_pct', 'relative_signed_error_pct'
]:
    if c in ALL.columns:
        ALL[c] = pd.to_numeric(ALL[c], errors='coerce')

TEST = ALL[ALL['split'].eq('test')].copy()

# One prediction row per method/anchor is required.
dup = TEST.duplicated(['anchor_key', 'method'], keep=False)
if dup.any():
    raise RuntimeError('Duplicate method/anchor predictions found in Notebook 03 outputs.')

# Exact-common support: exact same definition as Notebook 03.
wide_pred = TEST.pivot_table(
    index='anchor_key',
    columns='method',
    values='pred_remaining_duration_min',
    aggfunc='first'
)
for m in METHODS:
    if m not in wide_pred.columns:
        wide_pred[m] = np.nan

common_keys = set(
    wide_pred.dropna(subset=METHODS).index.astype(str)
)
TEST_COMMON = TEST[
    TEST['anchor_key'].astype(str).isin(common_keys)
].copy()

# Unique anchor-level metadata for all TEST anchors.
meta_cols = [
    'anchor_key', 'shared_event_uid', 'event_key', 'phase', 'room',
    'anchor_kind', 'true_remaining_duration_min',
    'remaining_trajectory_quality_score', 'event_duration_min'
]
ANCHOR_META = TEST_ANCHORS[meta_cols].drop_duplicates('anchor_key').copy()

# Cross-check Notebook 03 prediction truth/q against saved anchor metadata.
pred_meta = (
    TEST[['anchor_key', 'phase', 'room', 'event_key',
          'true_remaining_duration_min', 'remaining_trajectory_quality_score']]
    .drop_duplicates('anchor_key')
)
check = ANCHOR_META.merge(
    pred_meta,
    on='anchor_key',
    how='outer',
    suffixes=('_anchor', '_pred'),
    indicator=True
)
if not check['_merge'].eq('both').all():
    raise RuntimeError('Notebook 03 TEST anchors and authoritative TEST anchors do not match exactly.')

for c in ['true_remaining_duration_min', 'remaining_trajectory_quality_score']:
    a = pd.to_numeric(check[f'{c}_anchor'], errors='coerce').to_numpy(float)
    b = pd.to_numeric(check[f'{c}_pred'], errors='coerce').to_numpy(float)
    if not np.allclose(a, b, rtol=0.0, atol=1e-10, equal_nan=True):
        raise RuntimeError(f'Notebook 03 predictions disagree with authoritative anchors for {c}.')

# Build one-row-per-anchor common-support table.
COMMON_WIDE = ANCHOR_META[
    ANCHOR_META['anchor_key'].astype(str).isin(common_keys)
].copy()

for m in METHODS:
    s = (
        TEST.loc[TEST['method'].eq(m), ['anchor_key', 'pred_remaining_duration_min']]
        .drop_duplicates('anchor_key')
        .rename(columns={'pred_remaining_duration_min': m})
    )
    COMMON_WIDE = COMMON_WIDE.merge(s, on='anchor_key', how='left', validate='one_to_one')

if COMMON_WIDE[METHODS].isna().any().any():
    raise RuntimeError('Exact-common table contains a missing compared-method prediction.')

EXPECTED_COMMON = private_value('nb04.expected_common')

common_counts = (
    COMMON_WIDE.groupby('phase')
    .agg(
        anchors=('anchor_key', 'nunique'),
        events=('shared_event_uid', 'nunique'),
        rooms=('room', 'nunique'),
    )
    .reset_index()
)

print('Exact-common TEST support reconstructed independently:')
display(common_counts)

if STRICT_REFERENCE_AUDIT:
    for _, r in common_counts.iterrows():
        phase = r['phase']
        exp = EXPECTED_COMMON[phase]
        got = {k: int(r[k]) for k in ['anchors', 'events', 'rooms']}
        if got != exp:
            raise RuntimeError(
                f'{phase}: exact-common support differs from frozen executed reference. '
                f'Observed={got}, expected={exp}.'
            )
    print('Exact-common support audit: PASSED')


## 3. Reproduce reference metrics

For each prediction, \(e_i=\widehat{\tau}_i-\tau_i\); positive error means overestimated duration. Absolute and relative errors use the definitions in Notebook 03, including the five-minute relative-error denominator floor.

With `STRICT_REFERENCE_AUDIT=True`, reconstructed metrics must match both the saved result table and the privately supplied rounded reference values. A mismatch stops the analysis.


In [ ]:


# Exact Notebook-03 metric reproduction and dual reference check

def duration_metrics(g):
    true = pd.to_numeric(g['true_remaining_duration_min'], errors='coerce').to_numpy(float)
    pred = pd.to_numeric(g['pred_remaining_duration_min'], errors='coerce').to_numpy(float)
    good = np.isfinite(true) & np.isfinite(pred)
    true = true[good]
    pred = pred[good]

    if len(true) == 0:
        return pd.Series({
            'n': 0,
            'MAE_min': np.nan,
            'RMSE_min': np.nan,
            'Bias_min': np.nan,
            'MedianAE_min': np.nan,
            'P90AE_min': np.nan,
            'MeanRAE_pct': np.nan,
            'MedianRAE_pct': np.nan,
            'P90RAE_pct': np.nan,
            'RelativeBias_pct': np.nan,
        })

    e = pred - true
    ae = np.abs(e)
    denom = np.maximum(true, STEP_MIN)
    rae = 100.0 * ae / denom
    rse = 100.0 * e / denom

    return pd.Series({
        'n': len(e),
        'MAE_min': np.mean(ae),
        'RMSE_min': np.sqrt(np.mean(e ** 2)),
        'Bias_min': np.mean(e),
        'MedianAE_min': np.median(ae),
        'P90AE_min': np.percentile(ae, 90),
        'MeanRAE_pct': np.mean(rae),
        'MedianRAE_pct': np.median(rae),
        'P90RAE_pct': np.percentile(rae, 90),
        'RelativeBias_pct': np.mean(rse),
    })

recomputed_rows = []
for (phase, method), g in TEST_COMMON.groupby(['phase', 'method'], sort=False):
    row = duration_metrics(g).to_dict()
    row.update({'phase': phase, 'method': method})
    recomputed_rows.append(row)

MAIN_RECOMPUTED = pd.DataFrame(recomputed_rows)

# Compare with Notebook 03's own saved table at full available precision.
NB03_MAIN = pd.read_csv(NB03_MAIN_PATH)
metric_cols = [
    'n', 'MAE_min', 'RMSE_min', 'Bias_min', 'MedianAE_min', 'P90AE_min',
    'MeanRAE_pct', 'MedianRAE_pct', 'P90RAE_pct', 'RelativeBias_pct'
]

cmp = MAIN_RECOMPUTED.merge(
    NB03_MAIN,
    on=['phase', 'method'],
    suffixes=('_nb04', '_nb03'),
    validate='one_to_one'
)

for c in metric_cols:
    a = pd.to_numeric(cmp[f'{c}_nb04'], errors='coerce').to_numpy(float)
    b = pd.to_numeric(cmp[f'{c}_nb03'], errors='coerce').to_numpy(float)
    if not np.allclose(a, b, rtol=0.0, atol=1e-10, equal_nan=True):
        raise RuntimeError(f'Notebook 04 failed to reproduce Notebook 03 metric: {c}')

# Independent saved three-decimal audit against the executed notebook supplied with the
# paper.
EXPECTED_3DP = private_value('nb04.expected_3dp')

if STRICT_REFERENCE_AUDIT:
    for _, r in MAIN_RECOMPUTED.iterrows():
        key = (r['phase'], r['method'])
        exp = EXPECTED_3DP[key]
        for c, vexp in exp.items():
            v = float(r[c])
            # The reference check is intentionally at the precision printed by Notebook
            # 03.
            if round(v, 3) != round(float(vexp), 3):
                raise RuntimeError(
                    f'Frozen Notebook-03 audit failed for {key}, {c}: '
                    f'observed={v}, expected~={vexp}'
                )

print('Notebook 03 metric reproduction: PASSED')
display(
    MAIN_RECOMPUTED[
        ['phase', 'method'] + metric_cols
    ].sort_values(['phase', 'method']).round(3)
)

MAIN_RECOMPUTED.to_csv(
    TABLE_DIR / 'NB04_reproduced_Notebook03_exact_common_metrics.csv',
    index=False
)


## 4. Examine comparison coverage

Compare the full test set with the common cases retained for the three-model comparison. Report prediction-time, event and room counts, reasons for missing model outputs, and differences in observed duration and regularity between retained and excluded cases.

This describes the selection induced by the comparison rule without refitting models or changing their training support.


In [ ]:


# Exact-common support and selection audit

support = ANCHOR_META.copy()

for m in METHODS:
    p = (
        TEST.loc[TEST['method'].eq(m), ['anchor_key', 'pred_remaining_duration_min']]
        .drop_duplicates('anchor_key')
        .rename(columns={'pred_remaining_duration_min': f'pred__{m}'})
    )
    support = support.merge(p, on='anchor_key', how='left', validate='one_to_one')
    support[f'finite__{m}'] = np.isfinite(
        pd.to_numeric(support[f'pred__{m}'], errors='coerce')
    )

support['exact_common'] = support[[f'finite__{m}' for m in METHODS]].all(axis=1)

def missing_method_label(row):
    miss = [m for m in METHODS if not bool(row[f'finite__{m}'])]
    return 'none' if not miss else '+'.join(miss)

support['missing_methods'] = support.apply(missing_method_label, axis=1)

coverage_rows = []
for phase, g in support.groupby('phase', sort=False):
    gc = g[g['exact_common']]
    coverage_rows.append({
        'phase': phase,
        'full_test_anchors': int(g['anchor_key'].nunique()),
        'exact_common_anchors': int(gc['anchor_key'].nunique()),
        'retained_anchor_pct': 100.0 * gc['anchor_key'].nunique() / g['anchor_key'].nunique(),
        'excluded_anchors': int((~g['exact_common']).sum()),
        'full_test_events': int(g['shared_event_uid'].nunique()),
        'exact_common_events': int(gc['shared_event_uid'].nunique()),
        'retained_event_pct': 100.0 * gc['shared_event_uid'].nunique() / g['shared_event_uid'].nunique(),
        'full_test_rooms': int(g['room'].nunique()),
        'exact_common_rooms': int(gc['room'].nunique()),
    })

COVERAGE = pd.DataFrame(coverage_rows)
COVERAGE.to_csv(TABLE_DIR / 'PAPER_NB04_exact_common_support_summary.csv', index=False)

# Anchor-level common-vs-excluded characteristics.
anchor_char_rows = []
for (phase, common), g in support.groupby(['phase', 'exact_common'], sort=False):
    rem = pd.to_numeric(g['true_remaining_duration_min'], errors='coerce')
    q = pd.to_numeric(g['remaining_trajectory_quality_score'], errors='coerce')
    dur = pd.to_numeric(g['event_duration_min'], errors='coerce')
    kind = g['anchor_kind'].astype(str)
    anchor_char_rows.append({
        'phase': phase,
        'support_group': 'exact_common' if common else 'excluded',
        'anchors': int(g['anchor_key'].nunique()),
        'events': int(g['shared_event_uid'].nunique()),
        'rooms': int(g['room'].nunique()),
        'mean_true_remaining_min': float(rem.mean()),
        'median_true_remaining_min': float(rem.median()),
        'q25_true_remaining_min': float(rem.quantile(.25)),
        'q75_true_remaining_min': float(rem.quantile(.75)),
        'mean_event_duration_min': float(dur.mean()),
        'median_event_duration_min': float(dur.median()),
        'mean_q': float(q.mean()),
        'median_q': float(q.median()),
        'event_start_anchor_fraction': float(kind.eq('event_start_asof').mean()),
    })

ANCHOR_CHARACTERISTICS = pd.DataFrame(anchor_char_rows)
ANCHOR_CHARACTERISTICS.to_csv(
    TABLE_DIR / 'PAPER_NB04_common_vs_excluded_anchor_characteristics.csv',
    index=False
)

# Event-level retention and characteristics.
event_support = (
    support.groupby(
        ['phase', 'shared_event_uid', 'room'],
        as_index=False
    )
    .agg(
        test_anchors=('anchor_key', 'nunique'),
        common_anchors=('exact_common', 'sum'),
        event_duration_min=('event_duration_min', 'first'),
        median_true_remaining_min=('true_remaining_duration_min', 'median'),
        median_q=('remaining_trajectory_quality_score', 'median'),
    )
)
event_support['retained_anchor_fraction'] = (
    event_support['common_anchors'] / event_support['test_anchors']
)
event_support['support_status'] = np.select(
    [
        event_support['common_anchors'].eq(0),
        event_support['common_anchors'].eq(event_support['test_anchors']),
    ],
    ['excluded', 'fully_common'],
    default='partially_common'
)
event_support.to_csv(
    TABLE_DIR / 'PAPER_NB04_event_level_common_support_audit.csv',
    index=False
)

# Room-level support table.
ROOM_SUPPORT = (
    support.groupby(['phase', 'room'], as_index=False)
    .agg(
        test_anchors=('anchor_key', 'nunique'),
        common_anchors=('exact_common', 'sum'),
        test_events=('shared_event_uid', 'nunique'),
    )
)
ROOM_SUPPORT['retained_anchor_pct'] = (
    100.0 * ROOM_SUPPORT['common_anchors'] / ROOM_SUPPORT['test_anchors']
)
ROOM_SUPPORT.to_csv(
    TABLE_DIR / 'PAPER_NB04_room_level_common_support.csv',
    index=False
)

MISSING_REASON = (
    support.loc[~support['exact_common']]
    .groupby(['phase', 'missing_methods'], as_index=False)
    .agg(
        excluded_anchors=('anchor_key', 'nunique'),
        excluded_events=('shared_event_uid', 'nunique'),
        excluded_rooms=('room', 'nunique'),
    )
)
MISSING_REASON.to_csv(
    TABLE_DIR / 'PAPER_NB04_excluded_anchor_missing_method_audit.csv',
    index=False
)

print('Exact-common support summary:')
display(COVERAGE.round(3))

print('\nCommon vs excluded anchor characteristics:')
display(ANCHOR_CHARACTERISTICS.round(3))

print('\nExcluded-anchor missing-method audit:')
display(MISSING_REASON)


## 5. Compare prediction-time and event averages

Long events contribute more repeated predictions to a prediction-time average. Event-averaged MAE instead gives each event equal weight:

\[
\mathrm{MAE}_j=\frac{1}{n_j}\sum_{i\in j}|e_i|,
\qquad
\mathrm{MAE}_{\mathrm{event}}=\frac{1}{N}\sum_{j=1}^{N}\mathrm{MAE}_j.
\]

Both summaries are retained because they address different evaluation objectives. The code uses `MacroEvent` for event-averaged outputs.


In [ ]:


# Event-level metrics and event-averaged summaries

long_common = []
for method in METHODS:
    z = COMMON_WIDE[
        [
            'anchor_key', 'shared_event_uid', 'event_key', 'phase', 'room',
            'anchor_kind', 'true_remaining_duration_min',
            'remaining_trajectory_quality_score', method
        ]
    ].copy()
    z = z.rename(columns={method: 'pred_remaining_duration_min'})
    z['method'] = method
    z['error_min'] = (
        pd.to_numeric(z['pred_remaining_duration_min'], errors='coerce') -
        pd.to_numeric(z['true_remaining_duration_min'], errors='coerce')
    )
    z['absolute_error_min'] = z['error_min'].abs()
    denom = np.maximum(
        pd.to_numeric(z['true_remaining_duration_min'], errors='coerce'),
        STEP_MIN
    )
    z['relative_absolute_error_pct'] = 100.0 * z['absolute_error_min'] / denom
    z['relative_signed_error_pct'] = 100.0 * z['error_min'] / denom
    long_common.append(z)

COMMON_LONG = pd.concat(long_common, ignore_index=True)

event_rows = []
for (phase, event_uid, room, method), g in COMMON_LONG.groupby(
    ['phase', 'shared_event_uid', 'room', 'method'],
    sort=False
):
    e = pd.to_numeric(g['error_min'], errors='coerce').dropna().to_numpy(float)
    ae = np.abs(e)
    rae = pd.to_numeric(g['relative_absolute_error_pct'], errors='coerce').dropna().to_numpy(float)
    rse = pd.to_numeric(g['relative_signed_error_pct'], errors='coerce').dropna().to_numpy(float)
    event_rows.append({
        'phase': phase,
        'shared_event_uid': event_uid,
        'room': room,
        'method': method,
        'n_anchors': len(g),
        'Event_MAE_min': float(np.mean(ae)),
        'Event_RMSE_min': float(np.sqrt(np.mean(e ** 2))),
        'Event_Bias_min': float(np.mean(e)),
        'Event_MedianAE_min': float(np.median(ae)),
        'Event_MeanRAE_pct': float(np.mean(rae)),
        'Event_MedianRAE_pct': float(np.median(rae)),
        'Event_RelativeBias_pct': float(np.mean(rse)),
    })

PER_EVENT = pd.DataFrame(event_rows)
PER_EVENT.to_csv(TABLE_DIR / 'PAPER_NB04_per_event_metrics.csv', index=False)

macro_rows = []
for (phase, method), g in PER_EVENT.groupby(['phase', 'method'], sort=False):
    macro_rows.append({
        'phase': phase,
        'method': method,
        'n_events': int(g['shared_event_uid'].nunique()),
        'MacroEvent_MAE_min': float(g['Event_MAE_min'].mean()),
        'MedianEvent_MAE_min': float(g['Event_MAE_min'].median()),
        'MacroEvent_RMSE_min': float(g['Event_RMSE_min'].mean()),
        'MacroEvent_Bias_min': float(g['Event_Bias_min'].mean()),
        'MacroEvent_MeanRAE_pct': float(g['Event_MeanRAE_pct'].mean()),
        'MedianEvent_MeanRAE_pct': float(g['Event_MeanRAE_pct'].median()),
    })

MACRO = pd.DataFrame(macro_rows)
MACRO.to_csv(TABLE_DIR / 'PAPER_NB04_event_macro_metrics.csv', index=False)

print('Event-macro performance:')
display(MACRO.sort_values(['phase', 'method']).round(3))


## 6. Resample whole events

Within each operating mode, sample events with replacement and retain every prediction time from each selected event. Repeatedly selected events contribute repeatedly. Shared resamples preserve pairing between models and dependence among predictions from the same event.

The bootstrap summarizes error magnitudes, bias, relative errors, event averages and timing-tolerance rates. The configured default is 5,000 replicates with percentile 95% intervals.


In [ ]:


# Event-cluster bootstrap helpers

def _basic_metrics_numpy(true, pred):
    true = np.asarray(true, float)
    pred = np.asarray(pred, float)
    e = pred - true
    ae = np.abs(e)
    denom = np.maximum(true, STEP_MIN)
    rae = 100.0 * ae / denom
    return {
        'MAE_min': float(np.mean(ae)),
        'RMSE_min': float(np.sqrt(np.mean(e ** 2))),
        'Bias_min': float(np.mean(e)),
        'MedianAE_min': float(np.median(ae)),
        'MeanRAE_pct': float(np.mean(rae)),
        'MedianRAE_pct': float(np.median(rae)),
    }

def _build_phase_bootstrap_arrays(phase_df):
    z = phase_df.reset_index(drop=True).copy()
    events = z['shared_event_uid'].astype(str).drop_duplicates().tolist()
    event_to_code = {e: i for i, e in enumerate(events)}
    event_code = z['shared_event_uid'].astype(str).map(event_to_code).to_numpy(int)
    rows_by_event = [np.flatnonzero(event_code == i) for i in range(len(events))]

    true = pd.to_numeric(z['true_remaining_duration_min'], errors='coerce').to_numpy(float)
    preds = np.column_stack([
        pd.to_numeric(z[m], errors='coerce').to_numpy(float)
        for m in METHODS
    ])

    if not np.isfinite(true).all() or not np.isfinite(preds).all():
        raise RuntimeError(f'{phase_df["phase"].iloc[0]}: non-finite exact-common bootstrap data.')

    # Per-event metrics for macro summaries and tolerance macro summaries.
    event_mae = np.zeros((len(events), len(METHODS)), dtype=float)
    event_mean_rae = np.zeros_like(event_mae)
    event_tol = np.zeros((len(events), len(METHODS), len(TOLERANCES_MIN)), dtype=float)

    for ei, idx in enumerate(rows_by_event):
        yt = true[idx]
        for mi in range(len(METHODS)):
            yp = preds[idx, mi]
            e = yp - yt
            ae = np.abs(e)
            rae = 100.0 * ae / np.maximum(yt, STEP_MIN)
            event_mae[ei, mi] = ae.mean()
            event_mean_rae[ei, mi] = rae.mean()
            for ti, tol in enumerate(TOLERANCES_MIN):
                event_tol[ei, mi, ti] = 100.0 * np.mean(ae <= tol)

    # Room of each event; used by room-stratified event bootstrap.
    event_room = np.empty(len(events), dtype=object)
    for ei, idx in enumerate(rows_by_event):
        rooms = z.iloc[idx]['room'].astype(str).unique()
        if len(rooms) != 1:
            raise RuntimeError('A physical event maps to more than one room.')
        event_room[ei] = rooms[0]

    return {
        'df': z,
        'events': events,
        'event_code': event_code,
        'rows_by_event': rows_by_event,
        'event_room': event_room,
        'true': true,
        'preds': preds,
        'event_mae': event_mae,
        'event_mean_rae': event_mean_rae,
        'event_tol': event_tol,
    }

def _sample_event_codes(rng, arr, room_stratified=False):
    n_events = len(arr['events'])
    if not room_stratified:
        return rng.integers(0, n_events, size=n_events)

    sampled = []
    rooms = np.asarray(arr['event_room'], dtype=object)
    for room in sorted(pd.unique(rooms)):
        codes = np.flatnonzero(rooms == room)
        if len(codes) == 0:
            continue
        sampled.extend(
            rng.choice(codes, size=len(codes), replace=True).tolist()
        )
    return np.asarray(sampled, dtype=int)

def cluster_bootstrap_phase(phase, phase_df, n_boot, seed, room_stratified=False):
    arr = _build_phase_bootstrap_arrays(phase_df)
    rng = np.random.default_rng(seed)

    rows = []
    for b in range(n_boot):
        sampled_event_codes = _sample_event_codes(
            rng, arr, room_stratified=room_stratified
        )

        # Preserve multiplicity: if the same event is sampled twice, all its
        # anchors occur twice in the bootstrap sample.
        row_idx = np.concatenate([
            arr['rows_by_event'][ei]
            for ei in sampled_event_codes
        ])

        yt = arr['true'][row_idx]
        yp = arr['preds'][row_idx, :]

        for mi, method in enumerate(METHODS):
            met = _basic_metrics_numpy(yt, yp[:, mi])
            e = yp[:, mi] - yt
            ae = np.abs(e)

            rec = {
                'phase': phase,
                'replicate': b,
                'method': method,
                'bootstrap_scheme': (
                    'room_stratified_event'
                    if room_stratified
                    else 'event_cluster'
                ),
                **met,
                'MacroEvent_MAE_min': float(
                    arr['event_mae'][sampled_event_codes, mi].mean()
                ),
                'MacroEvent_MeanRAE_pct': float(
                    arr['event_mean_rae'][sampled_event_codes, mi].mean()
                ),
            }

            for ti, tol in enumerate(TOLERANCES_MIN):
                key = f'Within_{int(tol)}min_pct'
                rec[key] = float(100.0 * np.mean(ae <= tol))
                rec[f'MacroEvent_Within_{int(tol)}min_pct'] = float(
                    arr['event_tol'][sampled_event_codes, mi, ti].mean()
                )

            rows.append(rec)

    return pd.DataFrame(rows)

def percentile_ci(x, ci_level=CI_LEVEL):
    a = np.asarray(pd.Series(x).dropna(), float)
    alpha = 1.0 - ci_level
    return (
        float(np.quantile(a, alpha / 2.0)),
        float(np.quantile(a, 1.0 - alpha / 2.0)),
    )

def bootstrap_summary(boot_df, observed_df, metrics):
    rows = []
    for (phase, method), g in boot_df.groupby(['phase', 'method'], sort=False):
        obs = observed_df[
            (observed_df['phase'] == phase) &
            (observed_df['method'] == method)
        ].iloc[0]
        for metric in metrics:
            lo, hi = percentile_ci(g[metric])
            rows.append({
                'phase': phase,
                'method': method,
                'metric': metric,
                'observed': float(obs[metric]),
                'bootstrap_mean': float(g[metric].mean()),
                'bootstrap_se': float(g[metric].std(ddof=1)),
                'CI_level': CI_LEVEL,
                'CI_low': lo,
                'CI_high': hi,
                'bootstrap_scheme': g['bootstrap_scheme'].iloc[0],
            })
    return pd.DataFrame(rows)


In [ ]:


# Run primary event-cluster bootstrap

boot_frames = []

for pi, phase in enumerate(PHASES):
    phase_df = COMMON_WIDE[COMMON_WIDE['phase'].eq(phase)].copy()
    boot = cluster_bootstrap_phase(
        phase=phase,
        phase_df=phase_df,
        n_boot=N_BOOTSTRAP,
        seed=BOOTSTRAP_SEED + 1000 * pi,
        room_stratified=False,
    )
    boot_frames.append(boot)
    print(
        f'{phase}: completed {N_BOOTSTRAP} event-cluster bootstrap replicates '
        f'over {phase_df["shared_event_uid"].nunique()} physical events.'
    )

BOOT = pd.concat(boot_frames, ignore_index=True)
BOOT.to_csv(
    TABLE_DIR / 'NB04_event_cluster_bootstrap_replicates.csv.gz',
    index=False,
    compression='gzip'
)

# Observed pooled + macro table in one row per phase/method.
OBSERVED = MAIN_RECOMPUTED.merge(
    MACRO,
    on=['phase', 'method'],
    how='left',
    validate='one_to_one'
)

# Add observed operational tolerance metrics below before making the final
# bootstrap summary; primary uncertainty metrics can already be summarized.
PRIMARY_BOOT_METRICS = [
    'MAE_min',
    'RMSE_min',
    'Bias_min',
    'MedianAE_min',
    'MeanRAE_pct',
    'MedianRAE_pct',
    'MacroEvent_MAE_min',
    'MacroEvent_MeanRAE_pct',
]

BOOT_SUMMARY = bootstrap_summary(
    BOOT,
    OBSERVED,
    PRIMARY_BOOT_METRICS
)
BOOT_SUMMARY.to_csv(
    TABLE_DIR / 'PAPER_NB04_primary_metrics_event_cluster_bootstrap_CI.csv',
    index=False
)

print('Primary event-cluster bootstrap confidence intervals:')
display(BOOT_SUMMARY.round(3))


## 7. Calculate paired model differences

Compute each metric difference on the same bootstrap sample. For example,

\[
\Delta\mathrm{MAE}=\mathrm{MAE}_{\mathrm{NN}}-\mathrm{MAE}_{\mathrm{Direct\ ET}}.
\]

A negative error difference favors the first model. The reported fraction of bootstrap differences below zero is descriptive and is not a p-value.


In [ ]:


# Paired bootstrap differences

PAIRINGS = [
    ('Indirect_RoomPhase_ET', 'Direct_RoomPhase_ET', 'II_minus_I'),
    ('Unified_Derivative_NN', 'Direct_RoomPhase_ET', 'III_minus_I'),
    ('Unified_Derivative_NN', 'Indirect_RoomPhase_ET', 'III_minus_II'),
]

PAIR_METRICS = [
    'MAE_min',
    'RMSE_min',
    'MedianAE_min',
    'MedianRAE_pct',
    'MacroEvent_MAE_min',
    'MacroEvent_MeanRAE_pct',
]

paired_rows = []

for phase in PHASES:
    bp = BOOT[BOOT['phase'].eq(phase)]
    op = OBSERVED[OBSERVED['phase'].eq(phase)].set_index('method')

    for method_a, method_b, label in PAIRINGS:
        a = (
            bp[bp['method'].eq(method_a)]
            .sort_values('replicate')
            .reset_index(drop=True)
        )
        b = (
            bp[bp['method'].eq(method_b)]
            .sort_values('replicate')
            .reset_index(drop=True)
        )
        if not np.array_equal(a['replicate'].to_numpy(), b['replicate'].to_numpy()):
            raise RuntimeError('Paired bootstrap replicate alignment failed.')

        for metric in PAIR_METRICS:
            delta = a[metric].to_numpy(float) - b[metric].to_numpy(float)
            lo, hi = percentile_ci(delta)
            observed_delta = float(op.loc[method_a, metric] - op.loc[method_b, metric])

            paired_rows.append({
                'phase': phase,
                'comparison': label,
                'method_A': method_a,
                'method_B': method_b,
                'metric': metric,
                'observed_delta_A_minus_B': observed_delta,
                'CI_level': CI_LEVEL,
                'CI_low': lo,
                'CI_high': hi,
                'bootstrap_mean_delta': float(np.mean(delta)),
                'bootstrap_fraction_delta_lt_0': float(np.mean(delta < 0.0)),
                'bootstrap_fraction_delta_gt_0': float(np.mean(delta > 0.0)),
                'bootstrap_scheme': 'event_cluster',
            })

PAIRED = pd.DataFrame(paired_rows)
PAIRED.to_csv(
    TABLE_DIR / 'PAPER_NB04_paired_strategy_differences_event_cluster_bootstrap.csv',
    index=False
)

print('Paired event-cluster bootstrap differences:')
display(
    PAIRED[
        PAIRED['metric'].isin(['MAE_min', 'MacroEvent_MAE_min'])
    ].round(3)
)


## 8. Summarize timing tolerances

Report the fractions of common test predictions satisfying \(|\widehat{\tau}-\tau|\leq\delta\) for \(\delta\in\{5,10,15,20\}\) minutes. Also report underestimation and overestimation beyond each tolerance, with event-averaged summaries where available.

These are prediction-error measures, not measured control, energy or comfort outcomes.


In [ ]:


# Operational tolerance metrics

tol_rows = []
event_tol_rows = []

for (phase, method), g in COMMON_LONG.groupby(['phase', 'method'], sort=False):
    e = pd.to_numeric(g['error_min'], errors='coerce').to_numpy(float)
    ae = np.abs(e)

    for tol in TOLERANCES_MIN:
        tol_rows.append({
            'phase': phase,
            'method': method,
            'tolerance_min': tol,
            'n_anchors': len(e),
            'within_tolerance_pct': float(100.0 * np.mean(ae <= tol)),
            'underestimate_beyond_tolerance_pct': float(100.0 * np.mean(e < -tol)),
            'overestimate_beyond_tolerance_pct': float(100.0 * np.mean(e > tol)),
        })

        # Event-averaged version: percentage within tolerance is calculated
        # within each event, then averaged equally across events.
        per_event = (
            g.assign(within=np.abs(g['error_min']) <= tol)
            .groupby('shared_event_uid')['within']
            .mean()
        )
        event_tol_rows.append({
            'phase': phase,
            'method': method,
            'tolerance_min': tol,
            'n_events': int(len(per_event)),
            'macro_event_within_tolerance_pct': float(100.0 * per_event.mean()),
        })

TOL = pd.DataFrame(tol_rows)
EVENT_TOL = pd.DataFrame(event_tol_rows)

# Attach bootstrap confidence intervals for pooled and event-averaged within-tolerance
# percentages.
ci_rows = []
for (phase, method), g in BOOT.groupby(['phase', 'method'], sort=False):
    for tol in TOLERANCES_MIN:
        col = f'Within_{int(tol)}min_pct'
        macro_col = f'MacroEvent_Within_{int(tol)}min_pct'

        obs = TOL[
            (TOL['phase'].eq(phase)) &
            (TOL['method'].eq(method)) &
            (TOL['tolerance_min'].eq(tol))
        ].iloc[0]['within_tolerance_pct']

        obs_macro = EVENT_TOL[
            (EVENT_TOL['phase'].eq(phase)) &
            (EVENT_TOL['method'].eq(method)) &
            (EVENT_TOL['tolerance_min'].eq(tol))
        ].iloc[0]['macro_event_within_tolerance_pct']

        lo, hi = percentile_ci(g[col])
        mlo, mhi = percentile_ci(g[macro_col])

        ci_rows.append({
            'phase': phase,
            'method': method,
            'tolerance_min': tol,
            'within_tolerance_pct': float(obs),
            'within_CI_low': lo,
            'within_CI_high': hi,
            'macro_event_within_tolerance_pct': float(obs_macro),
            'macro_event_within_CI_low': mlo,
            'macro_event_within_CI_high': mhi,
        })

TOL_CI = pd.DataFrame(ci_rows)

TOL.to_csv(
    TABLE_DIR / 'PAPER_NB04_operational_error_bands_exact_common.csv',
    index=False
)
EVENT_TOL.to_csv(
    TABLE_DIR / 'PAPER_NB04_operational_error_bands_event_macro.csv',
    index=False
)
TOL_CI.to_csv(
    TABLE_DIR / 'PAPER_NB04_operational_error_bands_with_cluster_bootstrap_CI.csv',
    index=False
)

print('Operational error bands:')
display(TOL_CI.round(2))


## 9. Preserve room composition during resampling

Resample events within each room while retaining its observed event count. Compare these paired intervals with the primary event-bootstrap intervals to assess sensitivity to room composition in resampling. This does not give rooms equal weight or evaluate unseen rooms.


In [ ]:


# Optional room-stratified event bootstrap sensitivity

ROOM_STRATIFIED_BOOT = pd.DataFrame()
ROOM_STRATIFIED_PAIRED = pd.DataFrame()

if RUN_ROOM_STRATIFIED_BOOTSTRAP:
    rs_frames = []
    for pi, phase in enumerate(PHASES):
        phase_df = COMMON_WIDE[COMMON_WIDE['phase'].eq(phase)].copy()
        rs = cluster_bootstrap_phase(
            phase=phase,
            phase_df=phase_df,
            n_boot=N_BOOTSTRAP,
            seed=BOOTSTRAP_SEED + 10000 + 1000 * pi,
            room_stratified=True,
        )
        rs_frames.append(rs)
        print(f'{phase}: completed room-stratified event bootstrap.')

    ROOM_STRATIFIED_BOOT = pd.concat(rs_frames, ignore_index=True)
    ROOM_STRATIFIED_BOOT.to_csv(
        TABLE_DIR / 'NB04_room_stratified_event_bootstrap_replicates.csv.gz',
        index=False,
        compression='gzip'
    )

    rs_pair_rows = []
    for phase in PHASES:
        bp = ROOM_STRATIFIED_BOOT[
            ROOM_STRATIFIED_BOOT['phase'].eq(phase)
        ]
        op = OBSERVED[OBSERVED['phase'].eq(phase)].set_index('method')

        for method_a, method_b, label in PAIRINGS:
            a = (
                bp[bp['method'].eq(method_a)]
                .sort_values('replicate')
                .reset_index(drop=True)
            )
            b = (
                bp[bp['method'].eq(method_b)]
                .sort_values('replicate')
                .reset_index(drop=True)
            )

            for metric in ['MAE_min', 'MacroEvent_MAE_min']:
                delta = a[metric].to_numpy(float) - b[metric].to_numpy(float)
                lo, hi = percentile_ci(delta)
                rs_pair_rows.append({
                    'phase': phase,
                    'comparison': label,
                    'method_A': method_a,
                    'method_B': method_b,
                    'metric': metric,
                    'observed_delta_A_minus_B': float(
                        op.loc[method_a, metric] - op.loc[method_b, metric]
                    ),
                    'CI_level': CI_LEVEL,
                    'CI_low': lo,
                    'CI_high': hi,
                    'bootstrap_mean_delta': float(delta.mean()),
                    'bootstrap_fraction_delta_lt_0': float(np.mean(delta < 0)),
                    'bootstrap_scheme': 'room_stratified_event',
                })

    ROOM_STRATIFIED_PAIRED = pd.DataFrame(rs_pair_rows)
    ROOM_STRATIFIED_PAIRED.to_csv(
        TABLE_DIR / 'PAPER_NB04_paired_MAE_differences_room_stratified_bootstrap.csv',
        index=False
    )

    print('Room-stratified sensitivity — paired MAE differences:')
    display(ROOM_STRATIFIED_PAIRED.round(3))
else:
    print('Room-stratified bootstrap disabled by configuration.')


## 10. Plot evaluation summaries

Visualize prediction-time and event-averaged errors, paired differences, timing-tolerance curves and common-case retention. Plots use the saved comparisons without selecting or refitting models.


In [ ]:


# Paper-ready Notebook-04 figures

METHOD_LABELS = {
    'Direct_RoomPhase_ET': 'Direct ET',
    'Indirect_RoomPhase_ET': 'Indirect ET',
    'Unified_Derivative_NN': 'Unified NN',
}

# Figure A — pooled MAE and event-averaged MAE with event-bootstrap CI

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), constrained_layout=True)

for ax, metric, title in [
    (axes[0], 'MAE_min', 'Pooled-anchor MAE'),
    (axes[1], 'MacroEvent_MAE_min', 'Event-macro MAE'),
]:
    s = BOOT_SUMMARY[BOOT_SUMMARY['metric'].eq(metric)].copy()
    xpos = []
    xlabels = []
    x = 0
    for phase in PHASES:
        sp = s[s['phase'].eq(phase)]
        for method in METHODS:
            r = sp[sp['method'].eq(method)].iloc[0]
            y = r['observed']
            yerr = np.array([[y - r['CI_low']], [r['CI_high'] - y]])
            ax.errorbar(x, y, yerr=yerr, fmt='o', capsize=4)
            xpos.append(x)
            xlabels.append(f'{phase.capitalize()}\n{METHOD_LABELS[method]}')
            x += 1
        x += 0.6
    ax.set_xticks(xpos)
    ax.set_xticklabels(xlabels, rotation=25, ha='right')
    ax.set_ylabel('Error (min)')
    ax.set_title(title + f' — {int(CI_LEVEL*100)}% event-cluster bootstrap CI')
    ax.grid(axis='y', alpha=.25)

fig.savefig(
    FIG_DIR / 'PAPER_NB04_MAE_pooled_and_event_macro_with_cluster_bootstrap_CI.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

# Figure B — paired MAE differences with CI

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)

pair_labels = {
    'II_minus_I': 'Indirect ET − Direct ET',
    'III_minus_I': 'Unified NN − Direct ET',
    'III_minus_II': 'Unified NN − Indirect ET',
}

for ax, phase in zip(axes, PHASES):
    d = PAIRED[
        PAIRED['phase'].eq(phase) &
        PAIRED['metric'].eq('MAE_min')
    ].copy()
    y = np.arange(len(d))
    center = d['observed_delta_A_minus_B'].to_numpy(float)
    lo = d['CI_low'].to_numpy(float)
    hi = d['CI_high'].to_numpy(float)

    ax.errorbar(
        center, y,
        xerr=np.vstack([center - lo, hi - center]),
        fmt='o', capsize=4
    )
    ax.axvline(0.0, linestyle='--')
    ax.set_yticks(y)
    ax.set_yticklabels([pair_labels[x] for x in d['comparison']])
    ax.set_xlabel('Paired ΔMAE (min)')
    ax.set_title(phase.capitalize())
    ax.grid(axis='x', alpha=.25)

fig.savefig(
    FIG_DIR / 'PAPER_NB04_paired_MAE_differences_event_cluster_bootstrap_CI.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

# Figure C — operational tolerance curves

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.6), constrained_layout=True)

for ax, phase in zip(axes, PHASES):
    d = TOL_CI[TOL_CI['phase'].eq(phase)].copy()
    for method in METHODS:
        g = d[d['method'].eq(method)].sort_values('tolerance_min')
        ax.plot(
            g['tolerance_min'],
            g['within_tolerance_pct'],
            marker='o',
            label=METHOD_LABELS[method]
        )
    ax.set_xticks(TOLERANCES_MIN)
    ax.set_ylim(0, 100)
    ax.set_xlabel('Absolute-error tolerance (min)')
    ax.set_ylabel('Predictions within tolerance (%)')
    ax.set_title(phase.capitalize())
    ax.grid(alpha=.25)
    ax.legend(frameon=False)

fig.savefig(
    FIG_DIR / 'PAPER_NB04_operational_tolerance_curves_exact_common.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

# Figure D — exact-common retention

fig, ax = plt.subplots(figsize=(6.4, 4.6))
x = np.arange(len(COVERAGE))
ax.bar(x, COVERAGE['retained_anchor_pct'])
ax.set_xticks(x, COVERAGE['phase'].str.capitalize())
ax.set_ylim(0, 105)
ax.set_ylabel('TEST anchors retained in exact-common support (%)')
ax.set_title('Exact-common support retention')
ax.grid(axis='y', alpha=.25)

for i, r in COVERAGE.reset_index(drop=True).iterrows():
    ax.text(
        i,
        r['retained_anchor_pct'] + 1.5,
        f'{int(r["exact_common_anchors"])}/{int(r["full_test_anchors"])}',
        ha='center'
    )

fig.tight_layout()
fig.savefig(
    FIG_DIR / 'PAPER_NB04_exact_common_support_retention.png',
    dpi=300, bbox_inches='tight'
)
plt.show()

print('Notebook-04 figures saved to:', FIG_DIR)


## 11. Record analysis provenance

Save the input evaluation run, split and prediction-file fingerprints, comparison counts, bootstrap settings, tolerance values and output paths. Final checks confirm prediction coverage and consistency of comparison keys.


In [ ]:


# Final audit + manifest

# Guard 1 — shared NN must be finite on the complete saved TEST anchor set.
nn_test = TEST[TEST['method'].eq('Unified_Derivative_NN')].copy()
nn_finite = (
    nn_test.assign(
        finite=np.isfinite(
            pd.to_numeric(nn_test['pred_remaining_duration_min'], errors='coerce')
        )
    )
    .groupby('phase')['finite']
    .sum()
    .to_dict()
)

if STRICT_REFERENCE_AUDIT:
    if int(nn_finite.get('heating', -1)) != private_value('nb04.expected_full_test')['heating']['anchors']:
        raise RuntimeError('Unified NN heating full-TEST coverage changed.')
    if int(nn_finite.get('cooling', -1)) != private_value('nb04.expected_full_test')['cooling']['anchors']:
        raise RuntimeError('Unified NN cooling full-TEST coverage changed.')

# Guard 2 — exact-common keys must be identical for all methods.
for method in METHODS:
    keys = set(
        TEST.loc[
            TEST['method'].eq(method) &
            np.isfinite(pd.to_numeric(TEST['pred_remaining_duration_min'], errors='coerce')),
            'anchor_key'
        ].astype(str)
    )
    if not common_keys.issubset(keys):
        raise RuntimeError(f'{method}: exact-common key reconstruction failed.')

# Guard 3 — no training artifacts are created by Notebook 04.
# Only tables, figures, audits, and a manifest are written.

paper_outputs = {
    'support_summary': str(TABLE_DIR / 'PAPER_NB04_exact_common_support_summary.csv'),
    'selection_characteristics': str(TABLE_DIR / 'PAPER_NB04_common_vs_excluded_anchor_characteristics.csv'),
    'room_support': str(TABLE_DIR / 'PAPER_NB04_room_level_common_support.csv'),
    'event_macro_metrics': str(TABLE_DIR / 'PAPER_NB04_event_macro_metrics.csv'),
    'primary_bootstrap_CI': str(TABLE_DIR / 'PAPER_NB04_primary_metrics_event_cluster_bootstrap_CI.csv'),
    'paired_bootstrap_differences': str(TABLE_DIR / 'PAPER_NB04_paired_strategy_differences_event_cluster_bootstrap.csv'),
    'operational_error_bands': str(TABLE_DIR / 'PAPER_NB04_operational_error_bands_with_cluster_bootstrap_CI.csv'),
}

manifest = {
    'notebook': '04_statistical_uncertainty_support_and_operational_metrics_PAPER_FINAL',
    'role': 'post-hoc statistical/coverage extension; no model retraining',
    'authoritative_notebook03_run': str(NB03_RUN),
    'notebook03_run_id': NOTEBOOK03_RUN_ID,
    'notebook03_prediction_csv': str(PRED_PATH),
    'notebook03_prediction_sha256': INPUT_PREDICTION_SHA256,
    'shared_split_path': str(SPLIT_PATH),
    'shared_split_sha256': CURRENT_SPLIT_SHA256,
    'methods': METHODS,
    'primary_endpoint': 'remaining duration',
    'common_support_definition': (
        'TEST anchor retained iff all three PAPER-FINAL methods have finite predictions'
    ),
    'bootstrap_unit': 'physical event (shared_event_uid)',
    'bootstrap_replicates': N_BOOTSTRAP,
    'bootstrap_seed': BOOTSTRAP_SEED,
    'CI_level': CI_LEVEL,
    'bootstrap_scheme_primary': 'event-cluster bootstrap within phase',
    'bootstrap_scheme_sensitivity': (
        'room-stratified event bootstrap'
        if RUN_ROOM_STRATIFIED_BOOTSTRAP
        else 'disabled'
    ),
    'operational_tolerances_min': TOLERANCES_MIN,
    'full_test_support': COVERAGE[
        ['phase', 'full_test_anchors', 'full_test_events', 'full_test_rooms']
    ].to_dict('records'),
    'exact_common_support': COVERAGE[
        ['phase', 'exact_common_anchors', 'exact_common_events', 'exact_common_rooms']
    ].to_dict('records'),
    'paper_outputs': paper_outputs,
    'interpretation_guardrail': (
        'Intervals are conditional on the observed building, frozen TEST events, '
        'frozen predictions, and exact-common-support rule; they are not '
        'cross-building uncertainty and do not isolate causal component effects.'
    ),
}

manifest_path = OUT_ROOT / 'NB04_statistical_robustness_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, default=str)

# Compact final status table.
status = pd.DataFrame([
    {
        'check': 'Notebook 03 saved metrics reproduced exactly',
        'status': 'PASS',
    },
    {
        'check': 'Frozen three-decimal PAPER-FINAL metrics reproduced',
        'status': 'PASS' if STRICT_REFERENCE_AUDIT else 'NOT ENFORCED',
    },
    {
        'check': 'Shared split fingerprint matches Notebook 03',
        'status': 'PASS',
    },
    {
        'check': 'Unified NN full TEST support unchanged',
        'status': 'PASS',
    },
    {
        'check': 'Event is bootstrap resampling unit',
        'status': 'PASS',
    },
    {
        'check': 'No model retraining / no split modification',
        'status': 'PASS',
    },
])

status.to_csv(AUDIT_DIR / 'NB04_final_integrity_checks.csv', index=False)

print('\nFINAL NOTEBOOK-04 INTEGRITY STATUS')
display(status)
print('\nManifest:', manifest_path)
print('Output root:', OUT_ROOT)


## Output guide

The tables cover paired bootstrap differences, event-averaged metrics, comparison coverage, excluded cases and timing-tolerance rates. The manifest identifies their exact paths and source fingerprints.

Interpret paired intervals together with observed differences and the averaging scheme. An interval containing zero does not establish model equivalence. Report the selected comparison cohort and retain the single-building and event-resampling scope when discussing these results.
